In [1]:
import sqlite3
import pandas as pd
import os
import csv

In [2]:
DATABASE = "../../../../v04_verb-case_pattern/drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"

LINE_DATA_TABLE = "lines_class_info4"

FILTERED_CLASS_TABLE = "lines_class_info4_n20"

LARGE_DATA_FILE = "../../data/n20_examples_large_v01_2.csv"

LARGE_DATA_FILE_SORTED = "../../data/n20_examples_large_v01_2_sorted.csv"


## see teeb korrektse uue andmefaili v2

### andmetabelid

In [3]:
conn = sqlite3.connect(DATABASE)
cursor = conn.cursor()

### graafiku punktide info

In [4]:
query = f"""SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM {LINE_DATA_TABLE}
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
2,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
3,aerutama,,in,-9.965784,-,1,NaN,5.0,-,0,3,3,10,13
4,aevastama,,in,9.965784,-,1,1.0,6.0,-,1,0,1,6,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20935,õnnestuma,,ad,-1.971847,-,1313,331.0,2522.0,-,1022,4009,5031,17243,22274
20936,õppima,,in,5.233872,n90,530,463.0,985.0,0.0,5720,152,5872,5891,11763
20937,ütlema,,ad,-5.370614,n10,415,104.0,1162.0,0.0,295,12205,12500,9966,22466
20938,ütlema,,all,0.271387,n70,1131,189.0,2337.0,1.0,9354,7750,17104,40742,57846


### võtta ainult n20 tsooni lõksud

In [5]:
filtered_class = class_info[class_info["level"]=="n20"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [6]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [7]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
20933,võtma,,ad,-2.849458,n20,700,200.0,1565.0,0.00000,521,3755,4276,12272,16548
20734,mõjuma,,all,-3.021174,n20,586,106.0,1969.0,0.00000,186,1510,1696,6800,8496
20710,seletama,,all,-3.696855,n20,193,18.0,219.0,0.00000,31,402,433,1293,1726
20676,näitama,,all,-3.284561,n20,628,104.0,1018.0,0.00000,176,1715,1891,5226,7117
20657,lubama,,ad,-2.738744,n20,877,211.0,1427.0,0.00000,446,2977,3423,9171,12594
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20576,tuletama,,all,-3.147307,n20,217,30.0,250.0,0.00007,50,443,493,1487,1980
20411,väsima,,el,-3.779610,n20,116,15.0,559.0,0.00008,15,206,221,1015,1236
20152,hoolima,,el,-3.035091,n20,243,45.0,985.0,0.00008,66,541,607,3175,3782
20885,teatama,,el,-2.635637,n20,570,179.0,2193.0,0.00008,345,2144,2489,13915,16404


In [8]:
filtered_class.to_sql(FILTERED_CLASS_TABLE, conn, if_exists="replace", index=False)

534

### võtta spatial_obl tabelist näitelaused koos vajaliku infoga

In [8]:
# spatial obl_tabelist lõksud, mis on lines_class_info4_n20 tabelis
# iga lõksu kohta max 500 lemmat ja iga unikaalse lemma kohta 1 näide


query = f"""
WITH cleaned AS (
    -- Step 1 & 2: match subset table + remove rows with timex_tag NOT NULL
    SELECT
        d.head_id,
        d.row_loc as head_loc,
        d.form,
        d.lemma,
        d.verb,
        d.verb_compound,
        d.morph_case,
        d.sentence,
        d.sentence_id,
        d.timex_tag,
        d.ekilex_tag,
        d.ner_tag
    FROM spatial_obl AS d
    JOIN {FILTERED_CLASS_TABLE} AS s
      ON d.verb = s.verb
     AND d.verb_compound = s.verb_compound
     AND d.morph_case = s.morph_case
    WHERE d.timex_tag IS NULL
),

distinct_lemmas AS (
    -- Step 3 & 4: for each lemma, pick ONE sentence deterministically
    SELECT 
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case, lemma
            ORDER BY sentence_id   -- choose best or earliest sentence
        ) AS rn_per_lemma
    FROM cleaned
),

limited AS (
    -- Step 5: limit to 500 unique lemmas per (verb, verb_compound, morph_case)
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY verb, verb_compound, morph_case
            ORDER BY lemma        -- choose 500 lexicographically smallest lemmas
        ) AS rn_group_limit
    FROM distinct_lemmas
    WHERE rn_per_lemma = 1      -- keep only one sentence per lemma
)

-- Step 6: final output
SELECT
    sentence_id,
    head_id,
    head_loc, 
    verb,
    verb_compound,
    morph_case,
    lemma,
    form,
    sentence,
    timex_tag,
    ekilex_tag,
    ner_tag
FROM limited
WHERE rn_group_limit <= 500
ORDER BY verb, verb_compound, morph_case, lemma;

"""


spatial_obl_ex = pd.read_sql(query, conn)


In [9]:
spatial_obl_ex

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,timex_tag,ekilex_tag,ner_tag
0,1102812,1752299,5,abielluma,,ad,30.,30-ndal,""" Nad ei abiellu 30-ndal , nad ei abiellu sell...",None,None,None
1,12950469,20717333,5,abielluma,,ad,47.,47ndal,"Ja jumal tänatud , 47ndal abiellusime , 60 aas...",None,None,None
2,8438306,13524475,1,abielluma,,ad,54.,54ndal,"54ndal nad abiellusid , kuid sportlaste tavali...",None,None,None
3,6032687,9682040,1,abielluma,,ad,Angel,Angelil,Angelil ja Dion abiellusid 1994. aastal Montre...,None,None,PER
4,15245183,23810125,3,abielluma,,ad,Fiji,Fijil,Maikuus salaja Fijil abiellunud Tori Spelling ...,None,None,LOC
...,...,...,...,...,...,...,...,...,...,...,...,...
120595,13024450,20838635,6,ütlema,üles,ad,ärimees,ärimehel,Aasta aega patust puhas olnud ärimehel öelnud ...,None,alive,None
120596,11204665,17950183,3,ütlema,üles,ad,õhulainer,õhulaineril,Ka sel õhulaineril ütles üles parem mootor .,None,None,None
120597,5318479,8532955,21,ütlema,üles,ad,ühistulemus,ühistulemusel,"BMW boss Mario Thiessen tunnistas , et Montoya...",None,None,None
120598,7253489,11664353,1,ütlema,üles,ad,üks,Ühel,"Ühel ütlevad üles tootmissüsteemid , teisel na...",None,None,None


In [10]:
#spatial_obl_ex

,head_id,form,lemma,verb,verb_compound,morph_case,sentence,sentence_id,timex_tag,ekilex_tag,ner_tag
0,1752299,30-ndal,30.,abielluma,,ad,""" Nad ei abiellu 30-ndal , nad ei abiellu sell...",1102812,None,None,None
1,20717333,47ndal,47.,abielluma,,ad,"Ja jumal tänatud , 47ndal abiellusime , 60 aas...",12950469,None,None,None
2,13524475,54ndal,54.,abielluma,,ad,"54ndal nad abiellusid , kuid sportlaste tavali...",8438306,None,None,None
3,9682040,Angelil,Angel,abielluma,,ad,Angelil ja Dion abiellusid 1994. aastal Montre...,6032687,None,None,PER
4,23810125,Fijil,Fiji,abielluma,,ad,Maikuus salaja Fijil abiellunud Tori Spelling ...,15245183,None,None,LOC
...,...,...,...,...,...,...,...,...,...,...,...
120595,20838635,ärimehel,ärimees,ütlema,üles,ad,Aasta aega patust puhas olnud ärimehel öelnud ...,13024450,None,alive,None
120596,17950183,õhulaineril,õhulainer,ütlema,üles,ad,Ka sel õhulaineril ütles üles parem mootor .,11204665,None,None,None
120597,8532955,ühistulemusel,ühistulemus,ütlema,üles,ad,"BMW boss Mario Thiessen tunnistas , et Montoya...",5318479,None,None,None
120598,11664353,Ühel,üks,ütlema,üles,ad,"Ühel ütlevad üles tootmissüsteemid , teisel na...",7253489,None,None,None


In [10]:
# shuffle
df = spatial_obl_ex.sample(frac=1)

In [11]:
df.to_csv(LARGE_DATA_FILE, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [12]:
spatial_obl_ex.to_csv(LARGE_DATA_FILE_SORTED, encoding="utf-8", index = False,sep=",", quoting=csv.QUOTE_MINIMAL)

In [13]:
conn.close()

In [14]:
df2 = pd.read_csv(LARGE_DATA_FILE, encoding="utf-8", sep=",")

In [15]:
counts2 = df2.groupby(['verb','verb_compound', 'morph_case'], dropna=False).size().reset_index(name='count').sort_values('count', ascending=False)
counts2

,verb,verb_compound,morph_case,count
17,andma,NaN,el,500
16,andma,NaN,ad,500
529,üritama,NaN,ad,500
36,avaldama,NaN,ad,500
4,aitama,NaN,ad,500
...,...,...,...,...
300,potsatama,NaN,ad,18
453,tõstma,välja,ad,17
222,mattuma,NaN,ad,17
172,külmutama,NaN,ad,17
